# Add Citation data to datset
### This script fetches data from semantic scholar api, falls back to open alex api if semantic scholar did not found arxiv id.

In [ ]:
import pandas as pd
df = pd.read_parquet('dataset_without_citations.parquet.gzip')
df

In [3]:
import pandas as pd, requests, time, random, os
from tqdm import tqdm

# --- Config ---
MAX_RPS = 1  # 1 request per second for both APIs
MAX_RETRIES = 3
CHECKPOINT_EVERY = 500
CHECKPOINT_FILE = "citations_checkpoint.parquet"

def fetch_citation_count(arxiv_id, s2_api_key=None, email="you@email.com"):
    """Single-paper citation fetcher - Semantic Scholar first, then OpenAlex."""
    
    # Try Semantic Scholar first with exponential backoff
    s2_url = f"http://api.semanticscholar.org/graph/v1/paper/arXiv:{arxiv_id}"
    s2_headers = {"x-api-key": s2_api_key} if s2_api_key else {}
    
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(s2_url, params={"fields": "citationCount"}, 
                           headers=s2_headers, timeout=8)
            if r.status_code == 200:
                j = r.json()
                
                return {
                    "citations": j.get("citationCount", 0),
                    "source": "semantic_scholar",
                    "success": True,
                }
            
            if r.status_code == 404:
                print(f"[404 S2] not found for {arxiv_id}")
                continue
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 2))
                print(f"[429 S2] wait {wait}s for {arxiv_id} retrying for {attempt} time...")
                time.sleep(wait)
                continue
                
            else:
                print(f"[S2 {r.status_code}] error for {arxiv_id} at attempt {attempt}")
                wait = 2 ** attempt
                time.sleep(wait)
                continue    
                
        except Exception as e:
            wait = 2 ** attempt
            print(f"[S2 error] {arxiv_id}: {e}, retry in {wait}s")
            time.sleep(wait)
    
    # Fallback to OpenAlex after S2 exhausted
    print(f"[fallback→OpenAlex] {arxiv_id}")
    headers = {"User-Agent": "arxiv-citation-fetcher", "mailto": email}
    urls = [
        f"https://api.openalex.org/works/https://doi.org/10.48550/arXiv.{arxiv_id}",
        f"https://api.openalex.org/works/https://arxiv.org/abs/{arxiv_id}"
    ]
    
    for attempt in range(MAX_RETRIES):
        for url in urls:
            try:
                r = requests.get(url, headers=headers, timeout=8)
                
                if r.status_code == 200:
                    j = r.json()
                    return {
                        "citations": j.get("cited_by_count", 0),
                        "source": "openalex",
                        "success": True,
                    }
                if r.status_code == 404:
                    print(f"[404 OpenAlex] not found for {arxiv_id}")
                    continue
                if r.status_code == 429:
                    wait = 2 ** attempt
                    print(f"[429 OpenAlex] wait {wait}s for {arxiv_id}")
                    time.sleep(wait)
                    continue
                else:
                    print(f"[OpenAlex {r.status_code}] error for {arxiv_id} at attempt {attempt}")
                    wait = 2 ** attempt
                    time.sleep(wait)
                    continue    
                    
            except Exception as e:
                print(f"[OpenAlex error] {arxiv_id}: {e}")
                time.sleep(2 ** attempt)

    return {"citations": 0, "source": "Error", "success": False}


def add_citations_fast(df, id_col="id", email="you@email.com", s2_api_key=None):
    """Sequential fetcher at 1 req/s with checkpointing."""
    if "citations" not in df.columns: 
        df["citations"] = 0
    if "citation_source" not in df.columns: 
        df["citation_source"] = ""
    
    start = 0
    delay = 1 / MAX_RPS  # 1 second
    pbar = tqdm(range(start, len(df)), desc="Citations (1 req/s)")
    
    for i in pbar:
        arxiv_id = str(df.at[i, id_col])
        res = fetch_citation_count(arxiv_id, s2_api_key=s2_api_key, email=email)
        df.at[i, "citations"] = res["citations"]
        df.at[i, "citation_source"] = res["source"] or ""
        
        # 1 second delay plus tiny jitter
        time.sleep(delay + random.uniform(0.01, 0.03))
        
        # save checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df.to_parquet(CHECKPOINT_FILE, index=False)
            pbar.set_postfix({"checkpoint": f"{i+1}"})
    
    df.to_parquet("arxiv_with_citations.parquet", index=False)
    print("✅ Saved arxiv_with_citations.parquet")
    return df

### Optional - start from last checkpoint

In [ ]:
import pandas as pd
df = pd.read_parquet(CHECKPOINT_FILE)
left_df = df[df["citation_source"].isin(["", "Error"])] 
left_df = left_df.reset_index(drop=True) 
left_df

In [ ]:
df = add_citations_fast(left_df, id_col="arxiv_id", s2_api_key="insert_your_api_key_here", email="kamil.mleczko.2003@gmail.com")

### Combine checkpoints

In [5]:
df_1 = pd.read_parquet('citations_checkpoint_B1.parquet')
df_2 = pd.read_parquet('citations_checkpoint_B2.parquet')
df_3 = pd.read_parquet('citations_checkpoint_B3.parquet')
df_4 = pd.read_parquet('citations_checkpoint_B4.parquet')

### Save dataset locally to parquet or to parquet.gzip for export purposes

In [8]:
df_1_filtered = df_1[df_1["citation_source"] != ""]
df_2_filtered = df_2[df_2["citation_source"] != ""]
df_3_filtered = df_3[df_3["citation_source"] != ""]
df_4_filtered = df_4[df_4["citation_source"] != ""]
merged_df = pd.concat([df_1_filtered, df_2_filtered, df_3_filtered, df_4_filtered], ignore_index=True)
merged_df

,arxiv_id,title,abstract,categories_list,year,article_url,repo_url_list,authors_list,citations,citation_source
0,0705.4676,Recursive n-gram hashing is pairwise independe...,Many applications use sequences of n consecu...,"[cs.DB, cs.CL]",2016,https://arxiv.org/pdf/0705.4676.pdf,"[https://github.com/zhaoxiaofei/bindash, https...","[Lemire Daniel, Kaser Owen]",26,semantic_scholar
1,0804.4451,Dependence Structure Estimation via Copula,Dependence strucuture estimation is one of t...,"[cs.LG, cs.IR, stat.ME]",2019,https://arxiv.org/pdf/0804.4451.pdf,[https://github.com/majianthu/dse],"[Ma Jian, Sun Zengqi]",9,semantic_scholar
2,0811.3301,Faster Retrieval with a Two-Pass Dynamic-Time-...,The Dynamic Time Warping (DTW) is a popular ...,"[cs.DB, cs.CV]",2012,https://arxiv.org/pdf/0811.3301.pdf,[https://github.com/lemire/lbimproved],[Lemire Daniel],193,semantic_scholar
3,0902.4682,Lectures on Jacques Herbrand as a Logician,We give some lectures on the work on formal ...,"[cs.LO, cs.AI]",2014,https://arxiv.org/pdf/0902.4682.pdf,[https://github.com/thejohncrafter/flows],"[Wirth Claus-Peter, Siekmann Joerg, Benzmuelle...",4,semantic_scholar
4,0906.2027,Matrix Completion from Noisy Entries,"Given a matrix M of low-rank, we consider th...","[cs.LG, stat.ML]",2012,https://arxiv.org/pdf/0906.2027.pdf,[https://github.com/jasonsun0310/MatrixComplet...,"[Keshavan Raghunandan H., Montanari Andrea, Oh...",726,semantic_scholar
...,...,...,...,...,...,...,...,...,...,...
123134,2507.15351,One Step is Enough: Multi-Agent Reinforcement ...,On-demand ride-sharing platforms face the fund...,"[cs.AI, cs.ET, cs.MA]",2025,https://arxiv.org/pdf/2507.15351.pdf,[https://github.com/RS2002/OSPO],"[Zhao Zijian, Li Sen]",1,semantic_scholar
123135,2507.15454,ObjectGS: Object-aware Scene Reconstruction an...,3D Gaussian Splatting is renowned for its high...,"[cs.GR, cs.AI, cs.CV, cs.HC]",2025,https://arxiv.org/pdf/2507.15454.pdf,[https://github.com/RuijieZhu94/ObjectGS],"[Zhu Ruijie, Yu Mulin, Xu Linning, Jiang Lihan...",2,semantic_scholar
123136,2507.15641,Leveraging Context for Multimodal Fallacy Clas...,"In this paper, we present our submission to th...","[cs.CL, cs.AI]",2025,https://arxiv.org/pdf/2507.15641.pdf,[https://github.com/alessiopittiglio/mm-argfal...,[Pittiglio Alessio],0,semantic_scholar
123137,cs/0212008,Principal Manifolds and Nonlinear Dimension Re...,Nonlinear manifold learning from unorganized...,"[cs.LG, cs.AI]",2016,https://arxiv.org/pdf/cs/0212008.pdf,[https://github.com/gitr00ki3/vpw],"[Zhang Zhenyue, Zha Hongyuan]",52,semantic_scholar


In [9]:
merged_df["citation_source"].value_counts()

citation_source
semantic_scholar    122793
openalex               327
Error                   19
Name: count, dtype: int64

In [10]:
cleaned_df = merged_df[merged_df["citation_source"] != "Error"].copy()
cleaned_df["citation_source"].value_counts()

citation_source
semantic_scholar    122793
openalex               327
Name: count, dtype: int64

# Save finished dataset

In [ ]:
cleaned_df.to_parquet('arxiv_with_citations_dataset_full.parquet.gzip', index=False, compression='gzip')
cleaned_df.to_parquet('arxiv_with_citations_dataset_full.parquet', index=False)

citation_source
semantic_scholar    122793
openalex               327
Name: count, dtype: int64